<a href="https://colab.research.google.com/github/arurihelona/CrystalAIProject/blob/main/USEF_TSE_Libri2Mix_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [95]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [96]:
!pip install -q kagglehub librosa soundfile tqdm pandas scikit-learn matplotlib

In [97]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio, display
from tqdm.auto import tqdm

print("Setup complete")
print("PyTorch:", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

Setup complete
PyTorch: 2.11.0+cu130
GPU: Tesla T4


In [98]:
import kagglehub

print("kagglehub imported successfully")

kagglehub imported successfully


In [99]:
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")

print("Token found:", token is not None)

Token found: True


In [100]:
KAGGLE_DATASET_ID = "trngttrng12/libri2mix"

print(KAGGLE_DATASET_ID)

trngttrng12/libri2mix


In [101]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    KAGGLE_DATASET_ID,
    output_dir="/content/libri2mix"
)

print("Dataset path:")
print(dataset_path)

Dataset path:
/content/libri2mix


In [102]:
import os

BASE_DIR = "/content/libri2mix"

print("Dataset folders:")

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

Dataset folders:
libri2mix/
  .complete/
    datasets/
      trngttrng12/
        libri2mix/
          2/
  librimix_data/
    val/
      s2/
      s1/
      mix_both/
      mix_clean/
      noise/
    train/
      s2/
      s1/
      mix_both/
      mix_clean/
      noise/


In [103]:
from glob import glob

TRAIN_DIR = os.path.join(BASE_DIR, "librimix_data", "train")
VAL_DIR = os.path.join(BASE_DIR, "librimix_data", "val")

train_s1 = sorted(glob(os.path.join(TRAIN_DIR, "s1", "*.wav")))
train_s2 = sorted(glob(os.path.join(TRAIN_DIR, "s2", "*.wav")))
train_mix = sorted(glob(os.path.join(TRAIN_DIR, "mix_clean", "*.wav")))

val_s1 = sorted(glob(os.path.join(VAL_DIR, "s1", "*.wav")))
val_s2 = sorted(glob(os.path.join(VAL_DIR, "s2", "*.wav")))
val_mix = sorted(glob(os.path.join(VAL_DIR, "mix_clean", "*.wav")))

print("Training:")
print("s1:", len(train_s1))
print("s2:", len(train_s2))
print("mix:", len(train_mix))

print("\nValidation:")
print("s1:", len(val_s1))
print("s2:", len(val_s2))
print("mix:", len(val_mix))

Training:
s1: 12510
s2: 12510
mix: 12510

Validation:
s1: 1390
s2: 1390
mix: 1390


In [104]:
train_df = pd.DataFrame({
    "s1_path": train_s1,
    "s2_path": train_s2,
    "mix_path": train_mix
})

val_df = pd.DataFrame({
    "s1_path": val_s1,
    "s2_path": val_s2,
    "mix_path": val_mix
})

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

train_df.head()

Training examples: 12510
Validation examples: 1390


,s1_path,s2_path,mix_path
0,/content/libri2mix/librimix_data/train/s1/103-...,/content/libri2mix/librimix_data/train/s2/103-...,/content/libri2mix/librimix_data/train/mix_cle...
1,/content/libri2mix/librimix_data/train/s1/103-...,/content/libri2mix/librimix_data/train/s2/103-...,/content/libri2mix/librimix_data/train/mix_cle...
2,/content/libri2mix/librimix_data/train/s1/103-...,/content/libri2mix/librimix_data/train/s2/103-...,/content/libri2mix/librimix_data/train/mix_cle...
3,/content/libri2mix/librimix_data/train/s1/103-...,/content/libri2mix/librimix_data/train/s2/103-...,/content/libri2mix/librimix_data/train/mix_cle...
4,/content/libri2mix/librimix_data/train/s1/103-...,/content/libri2mix/librimix_data/train/s2/103-...,/content/libri2mix/librimix_data/train/mix_cle...


In [105]:
DEBUG_TRAIN_SIZE = 100
DEBUG_VAL_SIZE = 20

debug_train_df = train_df.iloc[:DEBUG_TRAIN_SIZE].reset_index(drop=True)
debug_val_df = val_df.iloc[:DEBUG_VAL_SIZE].reset_index(drop=True)

print("Debug training:", len(debug_train_df))
print("Debug validation:", len(debug_val_df))

Debug training: 100
Debug validation: 20


In [106]:
def load_audio(path):
    audio, sr = librosa.load(
        path,
        sr=16000,
        mono=True
    )

    audio = torch.tensor(audio, dtype=torch.float32)

    return audio

In [108]:
SAMPLE_RATE = 16000

SEGMENT_SECONDS = 4
SEGMENT_SAMPLES = SAMPLE_RATE * SEGMENT_SECONDS

print("Sample rate:", SAMPLE_RATE)
print("Segment duration:", SEGMENT_SECONDS, "seconds")
print("Segment samples:", SEGMENT_SAMPLES)

Sample rate: 16000
Segment duration: 4 seconds
Segment samples: 64000


In [109]:
sample_audio = load_audio(train_df.iloc[0]["s1_path"])

print("Shape:", sample_audio.shape)
print("Duration:", len(sample_audio) / SAMPLE_RATE, "seconds")

Shape: torch.Size([98880])
Duration: 6.18 seconds


In [110]:
def crop_or_pad_at_start(audio, start, length=SEGMENT_SAMPLES):
    end = start + length

    if start >= len(audio):
        return torch.zeros(length)

    segment = audio[start:min(end, len(audio))]

    if len(segment) < length:
        segment = F.pad(
            segment,
            (0, length - len(segment))
        )

    return segment

In [111]:
def get_reference_segment(
    speaker_audio,
    target_start,
    segment_length=SEGMENT_SAMPLES
):
    audio_length = len(speaker_audio)

    # Need enough audio for target + separate reference
    if audio_length < 2 * segment_length:
        return None

    possible_starts = []

    for start in range(
        0,
        audio_length - segment_length + 1,
        segment_length
    ):
        # Reference must NOT overlap target
        if (
            start + segment_length <= target_start
            or start >= target_start + segment_length
        ):
            possible_starts.append(start)

    if not possible_starts:
        return None

    reference_start = random.choice(possible_starts)

    reference = speaker_audio[
        reference_start:reference_start + segment_length
    ]

    return reference

In [112]:
class Libri2MixTSE_BothSpeakers(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        # Load all three audio signals
        mix = load_audio(row["mix_path"])
        speaker1_audio = load_audio(row["s1_path"])
        speaker2_audio = load_audio(row["s2_path"])

        minimum_length = 2 * SEGMENT_SAMPLES

        # Need enough audio for target + separate reference
        if (
            len(speaker1_audio) < minimum_length
            or len(speaker2_audio) < minimum_length
            or len(mix) < SEGMENT_SAMPLES
        ):
            new_index = random.randint(
                0,
                len(self.df) - 1
            )

            return self.__getitem__(new_index)

        # Make sure the selected segment exists
        max_start = min(
            len(speaker1_audio),
            len(speaker2_audio),
            len(mix)
        ) - SEGMENT_SAMPLES

        target_start = random.randint(
            0,
            max_start
        )

        # ------------------------------------------------
        # TARGETS
        # ------------------------------------------------

        mixture = crop_or_pad_at_start(
            mix,
            target_start
        )

        target_s1 = crop_or_pad_at_start(
            speaker1_audio,
            target_start
        )

        target_s2 = crop_or_pad_at_start(
            speaker2_audio,
            target_start
        )

        # ------------------------------------------------
        # REFERENCES
        # ------------------------------------------------

        reference_s1 = get_reference_segment(
            speaker1_audio,
            target_start
        )

        reference_s2 = get_reference_segment(
            speaker2_audio,
            target_start
        )

        return {
            "mixture": mixture.float(),

            "reference_s1": reference_s1.float(),
            "target_s1": target_s1.float(),

            "reference_s2": reference_s2.float(),
            "target_s2": target_s2.float()
        }

In [113]:
both_speakers_dataset = Libri2MixTSE_BothSpeakers(
    debug_train_df
)

print(
    "Number of training examples:",
    len(both_speakers_dataset)
)

Number of training examples: 100


In [114]:
sample = both_speakers_dataset[0]

for key, value in sample.items():
    print(key, value.shape)

mixture torch.Size([64000])
reference_s1 torch.Size([64000])
target_s1 torch.Size([64000])
reference_s2 torch.Size([64000])
target_s2 torch.Size([64000])


In [115]:
print("Mixture:")
display(
    Audio(
        sample["mixture"].numpy(),
        rate=SAMPLE_RATE
    )
)

print("Reference Speaker 1:")
display(
    Audio(
        sample["reference_s1"].numpy(),
        rate=SAMPLE_RATE
    )
)

print("Target Speaker 1:")
display(
    Audio(
        sample["target_s1"].numpy(),
        rate=SAMPLE_RATE
    )
)

print("Reference Speaker 2:")
display(
    Audio(
        sample["reference_s2"].numpy(),
        rate=SAMPLE_RATE
    )
)

print("Target Speaker 2:")
display(
    Audio(
        sample["target_s2"].numpy(),
        rate=SAMPLE_RATE
    )
)

Mixture:


Reference Speaker 1:


Target Speaker 1:


Reference Speaker 2:


Target Speaker 2:


In [116]:
both_speakers_loader = DataLoader(
    both_speakers_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

print("DataLoader created")

DataLoader created


In [117]:
batch = next(iter(both_speakers_loader))

for key, value in batch.items():
    print(key, value.shape)

mixture torch.Size([2, 64000])
reference_s1 torch.Size([2, 64000])
target_s1 torch.Size([2, 64000])
reference_s2 torch.Size([2, 64000])
target_s2 torch.Size([2, 64000])


In [118]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [119]:
mixture = batch["mixture"].to(device)

reference_s1 = batch["reference_s1"].to(device)
target_s1 = batch["target_s1"].to(device)

reference_s2 = batch["reference_s2"].to(device)
target_s2 = batch["target_s2"].to(device)

print("Mixture:", mixture.shape)
print("Reference S1:", reference_s1.shape)
print("Target S1:", target_s1.shape)
print("Reference S2:", reference_s2.shape)
print("Target S2:", target_s2.shape)

Mixture: torch.Size([2, 64000])
Reference S1: torch.Size([2, 64000])
Target S1: torch.Size([2, 64000])
Reference S2: torch.Size([2, 64000])
Target S2: torch.Size([2, 64000])


In [120]:
def si_sdr(
    estimate,
    target,
    eps=1e-8
):
    target = target - torch.mean(
        target,
        dim=-1,
        keepdim=True
    )

    estimate = estimate - torch.mean(
        estimate,
        dim=-1,
        keepdim=True
    )

    target_energy = torch.sum(
        target ** 2,
        dim=-1,
        keepdim=True
    ) + eps

    projection = (
        torch.sum(
            estimate * target,
            dim=-1,
            keepdim=True
        )
        / target_energy
    ) * target

    noise = estimate - projection

    ratio = (
        torch.sum(
            projection ** 2,
            dim=-1
        )
        /
        (
            torch.sum(
                noise ** 2,
                dim=-1
            ) + eps
        )
    )

    return 10 * torch.log10(
        ratio + eps
    )


def si_sdr_loss(estimate, target):
    return -si_sdr(
        estimate,
        target
    ).mean()

In [121]:
test_loss = si_sdr_loss(
    target_s1,
    target_s1
)

print("SI-SDR loss:", test_loss.item())

SI-SDR loss: -100.36508178710938
